# conservative_rebound_release_gate_v1

Notebook-first diagnostic for the state-objective overfit bottleneck. The previous target-generation loop produced fold-local labels, but the current target-position priority gives downtrend risk caps precedence over rebound floors. This notebook checks whether a conservative SOXX/SOXL spread-oscillation release gate can provide a sparse, high-precision target-semantics candidate before any BC/PPO retraining.

Governance fields:

- **Canonical baseline**: `v19reg_v22_budget_usage_bull_guard_v1` runtime composer.
- **Direct comparison baseline**: `fold_local_joint_reward_targets_v1` plus `fold_local_rebound_priority_conflict_v1`.
- **Role**: `diagnose`, not promotion.
- **Changed layer**: `model_selection`, `reward`, `feature`; no PPO training and no execution promotion in this notebook.
- **Target window**: `wf_2022_train` 2020H1 event split for release-state recognition, with post-event false-positive audit on the remaining `wf_2022_train` rows.
- **Accepted trade-off**: lower recall is acceptable only if precision and false-positive lift are tightly controlled, because the objective is to avoid overfit while restoring erased rebound target lift.
- **Hypothesis**: SOXX/SOXL spread-oscillation features can identify a small subset of rebound-release rows with enough precision to justify revising target semantics before weighted BC/PPO.
- **Failure mode**: a high-recall release head that fires outside the event window would be another overfit/false-positive amplifier; it must stay diagnostic only.

External references used for the design:

- Microsoft Qlib: https://github.com/microsoft/qlib and paper https://arxiv.org/abs/2009.11189
- FinRL: https://github.com/AI4Finance-Foundation/FinRL and paper https://arxiv.org/abs/2011.09607
- Focal Loss for sparse/imbalanced events: https://arxiv.org/abs/1708.02002
- Prioritized Experience Replay for sparse/high-error transitions: https://arxiv.org/abs/1511.05952
- Probability of Backtest Overfitting / CSCV: https://doi.org/10.21314/JCF.2016.322

Closest historical success case: the retained selective-release line improved recovery/proxy diagnostics before walk-forward exposed execution absorption. Closest historical failure case: direct SOXX retrain and broad momentum release labels did not pass gates; high-recall state heads created false-positive release risk. This run is not a duplicate because it does **not** train a new model or promote a SOXX factor; it audits target semantics and a sparse release-gate candidate first.

In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from scripts.evaluate_factor_pack import _load_ohlcv_frames
from src.soxl_quant.app.representation_diagnostics import (
    StateHeadTargetRecipeConfig,
    score_state_head_target_recipes,
)
from src.soxl_quant.app.state_objective_training_targets import (
    audit_state_objective_target_lift,
    summarize_state_objective_target_lift_audit,
)
from src.soxl_quant.data import engineer_feature_frame
from src.soxl_quant.policy.policy_builder import state_objective_target_position_from_row

EXPERIMENT = 'conservative_rebound_release_gate_v1'
CARD = ROOT / 'runtime/model_selection/card' / EXPERIMENT
SOURCE_CARD = ROOT / 'runtime/model_selection/card/fold_local_joint_reward_targets_v1'
PRIOR_CARD = ROOT / 'runtime/model_selection/card/fold_local_rebound_priority_conflict_v1'
TARGETS_PATH = SOURCE_CARD / 'fold_local_joint_targets.csv'
CREATED_AT = datetime.now(timezone.utc).isoformat()
CARD.mkdir(parents=True, exist_ok=True)


def json_default(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (pd.Timestamp,)):
        return value.isoformat()
    if pd.isna(value):
        return None
    raise TypeError(f'Object of type {type(value).__name__} is not JSON serializable')


def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, default=json_default) + '\n', encoding='utf-8')


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
targets = pd.read_csv(TARGETS_PATH, parse_dates=['datetime'])

target_lift_audit = audit_state_objective_target_lift(targets, group_cols=('fold',))
target_lift_summary = summarize_state_objective_target_lift_audit(target_lift_audit)

target_lift_audit.to_csv(CARD / 'target_lift_audit.csv', index=False)
write_json(CARD / 'target_lift_summary.json', target_lift_summary)

display(target_lift_audit)
target_lift_summary


,fold,rows,rebound_rows,downtrend_rows,rebound_downtrend_rows,current_positive_rebound_lift_rows,candidate_release_positive_lift_rows,mean_current_effective_lift,mean_candidate_release_lift,max_current_effective_lift,max_candidate_release_lift,lift_semantics_gate,release_candidate_gate,target_position_priority,min_lift
0,wf_2022_train,505,66,234,57,0,57,0.0,0.329555,0.0,0.556976,False,True,risk,1.000000e-09
1,wf_2023_2024_train,756,66,478,57,0,57,0.0,0.329555,0.0,0.556976,False,True,risk,1.000000e-09


{'decision': 'state_objective_target_lift_blocked_no_training',
 'training_allowed': False,
 'lift_semantics_gate': False,
 'audited_groups': 2,
 'blocked_groups': 2,
 'rebound_rows': 132,
 'current_positive_rebound_lift_rows': 0,
 'candidate_release_positive_lift_rows': 114,
 'rebound_downtrend_rows': 114,
 'max_candidate_release_lift': 0.5569758652,
 'next_mechanism': 'revise_state_objective_target_semantics_before_weighted_bc'}

The audit must block direct weighted BC/PPO if current target semantics contain no effective rebound lift. A release gate is useful only as a target-semantics candidate when it has heldout precision and a clean post-event false-positive audit.

In [3]:
soxl, soxx = _load_ohlcv_frames()
features, _ = engineer_feature_frame(
    soxl,
    soxx,
    feature_packs=('transition_signals', 'soxx_signals', 'soxx_spread_oscillation'),
)
features = features.reset_index().rename(columns={'index': 'datetime'})
features['datetime'] = pd.to_datetime(features['datetime'])
if 'instrument' in features.columns:
    features = features[features['instrument'].astype(str).str.upper().eq('SOXL')].copy()

feature_cols = [
    'mom_5',
    'mom_21',
    'drawdown_63',
    'realized_vol_21',
    'vol_of_vol_21',
    'REL_VOLUME_RATIO',
    'PRICE_RATIO_MOM',
    'SOXX_BIAS_10',
    'SIGNED_TE_REVERSION_10',
    'TRANS_TE_ZSCORE',
    'SOXL_SOXX_RATIO_Z_252',
    'SOXL_SOXX_RATIO_WIDEN_5',
    'SOXL_SOXX_SPREAD_VOL_Z_21',
    'SOXL_SOXX_CHOP_DIVERGENCE_21',
]
feature_frame = features[['datetime', *feature_cols]].copy()

merge_base = targets.drop(columns=[c for c in feature_cols if c in targets.columns], errors='ignore')
frame = merge_base.merge(feature_frame, on='datetime', how='left')
for col in ['is_rebound_cluster', 'is_downtrend_state']:
    frame[col] = frame[col].fillna(False).astype(bool)

frame['current_target_position'] = [
    state_objective_target_position_from_row(row)
    for row in frame.to_dict('records')
]
frame['no_rebound_position'] = np.where(
    frame['is_downtrend_state'],
    frame['risk_target_position'],
    frame['teacher_position'],
)
frame['candidate_release_position'] = np.where(
    frame['is_downtrend_state'],
    np.maximum(frame['risk_target_position'], frame['rebound_target_position']),
    np.maximum(frame['teacher_position'], frame['rebound_target_position']),
)
frame['true_release_position'] = np.where(
    frame['is_rebound_cluster'],
    frame['candidate_release_position'],
    frame['current_target_position'],
)
frame['candidate_release_lift'] = frame['candidate_release_position'] - frame['no_rebound_position']
frame['true_release_lift'] = frame['true_release_position'] - frame['no_rebound_position']

coverage = {
    'rows': int(len(frame)),
    'source_targets': str(TARGETS_PATH.relative_to(ROOT)),
    'date_start': str(frame['datetime'].min().date()),
    'date_end': str(frame['datetime'].max().date()),
    'folds': sorted(frame['fold'].dropna().astype(str).unique().tolist()),
    'rebound_rows': int(frame['is_rebound_cluster'].sum()),
    'rebound_downtrend_rows': int((frame['is_rebound_cluster'] & frame['is_downtrend_state']).sum()),
    'feature_na_rate_max': float(frame[feature_cols].isna().mean().max()),
}
write_json(CARD / 'coverage.json', coverage)
coverage


{'rows': 1261,
 'source_targets': 'runtime/model_selection/card/fold_local_joint_reward_targets_v1/fold_local_joint_targets.csv',
 'date_start': '2020-01-02',
 'date_end': '2022-12-30',
 'folds': ['wf_2022_train', 'wf_2023_2024_train'],
 'rebound_rows': 132,
 'rebound_downtrend_rows': 114,
 'feature_na_rate_max': 0.0}

In [4]:
event_frame = frame[
    (frame['fold'].astype(str) == 'wf_2022_train')
    & (frame['datetime'] >= pd.Timestamp('2020-01-01'))
    & (frame['datetime'] <= pd.Timestamp('2020-06-30'))
].copy()
event_frame['event_fold'] = event_frame['datetime'].dt.to_period('Q').astype(str)

event_summary = (
    event_frame.groupby('event_fold')
    .agg(
        rows=('datetime', 'count'),
        rebound_rows=('is_rebound_cluster', 'sum'),
        non_rebound_rows=('is_rebound_cluster', lambda s: int((~s.astype(bool)).sum())),
        downtrend_rows=('is_downtrend_state', 'sum'),
        release_lift_rows=('true_release_lift', lambda s: int((s.abs() > 1e-9).sum())),
        start=('datetime', 'min'),
        end=('datetime', 'max'),
    )
    .reset_index()
)
event_summary.to_csv(CARD / 'event_fold_summary.csv', index=False)
event_summary


,event_fold,rows,rebound_rows,non_rebound_rows,downtrend_rows,release_lift_rows,start,end
0,2020Q1,62,10,52,35,10,2020-01-02,2020-03-31
1,2020Q2,63,56,7,50,47,2020-04-01,2020-06-30


In [5]:
feature_sets = {
    'momentum_drawdown': ['mom_5', 'mom_21', 'drawdown_63'],
    'momentum_soxx_signals': ['mom_5', 'mom_21', 'drawdown_63', 'REL_VOLUME_RATIO', 'PRICE_RATIO_MOM', 'SOXX_BIAS_10'],
    'signed_te_combo': ['mom_5', 'mom_21', 'drawdown_63', 'SIGNED_TE_REVERSION_10', 'TRANS_TE_ZSCORE'],
    'soxx_spread_oscillation': [
        'SOXL_SOXX_RATIO_Z_252',
        'SOXL_SOXX_RATIO_WIDEN_5',
        'SOXL_SOXX_SPREAD_VOL_Z_21',
        'SOXL_SOXX_CHOP_DIVERGENCE_21',
    ],
    'all_candidate': [
        'mom_5',
        'mom_21',
        'drawdown_63',
        'realized_vol_21',
        'vol_of_vol_21',
        'REL_VOLUME_RATIO',
        'PRICE_RATIO_MOM',
        'SOXX_BIAS_10',
        'SIGNED_TE_REVERSION_10',
        'TRANS_TE_ZSCORE',
        'SOXL_SOXX_RATIO_Z_252',
        'SOXL_SOXX_RATIO_WIDEN_5',
        'SOXL_SOXX_SPREAD_VOL_Z_21',
        'SOXL_SOXX_CHOP_DIVERGENCE_21',
    ],
}

conservative_config = StateHeadTargetRecipeConfig(
    min_recall=0.15,
    min_precision=0.75,
    max_false_positive_rate=0.05,
    min_effective_lift=0.01,
    max_effective_false_positive_rate=0.02,
    max_false_positive_lift_mean=0.02,
    max_false_positive_lift_max=0.20,
    max_target_mae=0.20,
    max_rebound_mae=0.35,
    min_covered_folds=2,
)

recipe = score_state_head_target_recipes(
    event_frame,
    label_col='is_rebound_cluster',
    fold_col='event_fold',
    true_target_col='true_release_position',
    default_target_col='no_rebound_position',
    state_target_col='candidate_release_position',
    feature_sets=feature_sets,
    thresholds=(-1.0, -0.5, 0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5),
    min_positive_train=5,
    min_positive_eval=5,
    config=conservative_config,
)
recipe.recipe_scores.to_csv(CARD / 'conservative_recipe_scores.csv', index=False)
recipe.row_scores.to_csv(CARD / 'conservative_recipe_row_scores.csv', index=False)
recipe.recipe_scores.head(20)


,feature_set,threshold,samples,positives,negatives,covered_folds,true_positive,false_positive,false_negative,true_negative,...,false_positive_rate,effective_false_positive,effective_false_positive_rate,false_positive_lift_mean,false_positive_lift_max,target_mae,rebound_target_mae,non_rebound_target_mae,recipe_gate,recipe_loss
0,soxx_spread_oscillation,0.50,125,66,59,2,17,1,49,58,...,0.016949,0,0.0,0.0,0.0,0.142244,0.269402,0.0,True,0.000000
1,soxx_spread_oscillation,0.75,125,66,59,2,15,0,51,59,...,0.000000,0,0.0,0.0,0.0,0.146964,0.278342,0.0,True,0.000000
2,all_candidate,1.00,125,66,59,2,13,2,53,57,...,0.033898,0,0.0,0.0,0.0,0.153205,0.290161,0.0,True,0.000000
3,soxx_spread_oscillation,1.00,125,66,59,2,12,0,54,59,...,0.000000,0,0.0,0.0,0.0,0.153285,0.290312,0.0,True,0.000000
4,momentum_drawdown,0.25,125,66,59,2,12,1,54,58,...,0.016949,0,0.0,0.0,0.0,0.154805,0.293191,0.0,True,0.000000
5,soxx_spread_oscillation,1.25,125,66,59,2,11,0,55,59,...,0.000000,0,0.0,0.0,0.0,0.156405,0.296222,0.0,True,0.000000
6,all_candidate,1.25,125,66,59,2,10,0,56,59,...,0.000000,0,0.0,0.0,0.0,0.158005,0.299252,0.0,True,0.000000
7,momentum_drawdown,0.50,125,66,59,2,10,0,56,59,...,0.000000,0,0.0,0.0,0.0,0.158005,0.299252,0.0,True,0.000000
8,soxx_spread_oscillation,0.25,125,66,59,2,19,4,47,55,...,0.067797,0,0.0,0.0,0.0,0.136596,0.258705,0.0,False,0.035593
9,all_candidate,0.50,125,66,59,2,16,4,50,55,...,0.067797,0,0.0,0.0,0.0,0.146885,0.278191,0.0,False,0.035593


In [6]:
def score_full_period_with_event_centroid(feature_set: str, threshold: float) -> pd.DataFrame:
    features_for_set = feature_sets[feature_set]
    train = event_frame.copy()
    train_values = train[features_for_set].apply(pd.to_numeric, errors='coerce')
    mean = train_values.mean(axis=0)
    std = train_values.std(axis=0, ddof=0).replace(0.0, 1.0).fillna(1.0)
    train_z = ((train_values - mean) / std).replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=float)
    labels = train['is_rebound_cluster'].to_numpy(dtype=bool)
    pos_center = train_z[labels].mean(axis=0)
    neg_center = train_z[~labels].mean(axis=0)

    scored = frame[frame['fold'].astype(str) == 'wf_2022_train'].copy()
    eval_values = scored[features_for_set].apply(pd.to_numeric, errors='coerce')
    eval_z = ((eval_values - mean) / std).replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=float)
    scored['state_score'] = np.linalg.norm(eval_z - neg_center, axis=1) - np.linalg.norm(eval_z - pos_center, axis=1)
    scored['predicted_release'] = scored['state_score'] >= float(threshold)
    scored['period'] = np.where(scored['datetime'] <= pd.Timestamp('2020-06-30'), 'event_2020h1', 'post_event')
    return scored

recipe_pass = recipe.recipe_scores[recipe.recipe_scores['recipe_gate'].astype(bool)].copy()
post_rows = []
row_outputs = []
for candidate in recipe_pass.to_dict('records'):
    scored = score_full_period_with_event_centroid(str(candidate['feature_set']), float(candidate['threshold']))
    scored['feature_set'] = str(candidate['feature_set'])
    scored['threshold'] = float(candidate['threshold'])
    row_outputs.append(scored[[
        'datetime', 'feature_set', 'threshold', 'period', 'state_score', 'predicted_release',
        'is_rebound_cluster', 'candidate_release_lift', 'no_rebound_position', 'candidate_release_position',
    ]])
    for period, group in scored.groupby('period', sort=True):
        labels = group['is_rebound_cluster'].astype(bool)
        preds = group['predicted_release'].astype(bool)
        negatives = ~labels
        tp = int((labels & preds).sum())
        fp = int((negatives & preds).sum())
        fn = int((labels & ~preds).sum())
        tn = int((negatives & ~preds).sum())
        effective_fp = int((negatives & preds & (group['candidate_release_lift'].abs() > 0.01)).sum())
        post_rows.append({
            'feature_set': str(candidate['feature_set']),
            'threshold': float(candidate['threshold']),
            'period': period,
            'rows': int(len(group)),
            'true_positive': tp,
            'false_positive': fp,
            'false_negative': fn,
            'true_negative': tn,
            'false_positive_rate': fp / max(fp + tn, 1),
            'effective_false_positive_rows': effective_fp,
            'prediction_rate': float(preds.mean()) if len(group) else 0.0,
            'mean_candidate_lift_on_effective_fp': float(
                group.loc[negatives & preds & (group['candidate_release_lift'].abs() > 0.01), 'candidate_release_lift'].mean()
            ) if effective_fp else 0.0,
        })

post_event_audit = pd.DataFrame(post_rows)
post_event_row_scores = pd.concat(row_outputs, ignore_index=True) if row_outputs else pd.DataFrame()
post_event_audit.to_csv(CARD / 'post_event_false_positive_audit.csv', index=False)
post_event_row_scores.to_csv(CARD / 'post_event_release_row_scores.csv', index=False)
post_event_audit


,feature_set,threshold,period,rows,true_positive,false_positive,false_negative,true_negative,false_positive_rate,effective_false_positive_rows,prediction_rate,mean_candidate_lift_on_effective_fp
0,soxx_spread_oscillation,0.50,event_2020h1,125,48,0,18,59,0.0,0,0.384,0.0
1,soxx_spread_oscillation,0.50,post_event,380,0,0,0,380,0.0,0,0.000,0.0
2,soxx_spread_oscillation,0.75,event_2020h1,125,40,0,26,59,0.0,0,0.320,0.0
3,soxx_spread_oscillation,0.75,post_event,380,0,0,0,380,0.0,0,0.000,0.0
4,all_candidate,1.00,event_2020h1,125,40,0,26,59,0.0,0,0.320,0.0
5,all_candidate,1.00,post_event,380,0,0,0,380,0.0,0,0.000,0.0
6,soxx_spread_oscillation,1.00,event_2020h1,125,26,0,40,59,0.0,0,0.208,0.0
7,soxx_spread_oscillation,1.00,post_event,380,0,0,0,380,0.0,0,0.000,0.0
8,momentum_drawdown,0.25,event_2020h1,125,47,0,19,59,0.0,0,0.376,0.0
9,momentum_drawdown,0.25,post_event,380,0,0,0,380,0.0,0,0.000,0.0


In [7]:
def post_event_pass(feature_set: str, threshold: float) -> bool:
    if post_event_audit.empty:
        return False
    rows = post_event_audit[
        (post_event_audit['feature_set'].astype(str) == str(feature_set))
        & (post_event_audit['threshold'].astype(float) == float(threshold))
        & (post_event_audit['period'].astype(str) == 'post_event')
    ]
    if rows.empty:
        return False
    row = rows.iloc[0]
    return bool(
        float(row['false_positive_rate']) <= 0.05
        and int(row['effective_false_positive_rows']) == 0
        and float(row['prediction_rate']) <= 0.05
    )

if recipe_pass.empty:
    selected = {}
else:
    ranked = recipe_pass.copy()
    ranked['post_event_gate'] = [
        post_event_pass(str(row.feature_set), float(row.threshold))
        for row in ranked.itertuples(index=False)
    ]
    ranked['prefer_soxx_spread'] = ranked['feature_set'].astype(str).eq('soxx_spread_oscillation')
    ranked = ranked.sort_values(
        ['post_event_gate', 'prefer_soxx_spread', 'false_positive_rate', 'state_precision', 'state_recall', 'feature_set', 'threshold'],
        ascending=[False, False, True, False, False, True, True],
    ).reset_index(drop=True)
    selected = ranked.iloc[0].to_dict()
    ranked.to_csv(CARD / 'selected_release_gate_ranking.csv', index=False)

selected


{'feature_set': 'soxx_spread_oscillation',
 'threshold': 0.75,
 'samples': 125,
 'positives': 66,
 'negatives': 59,
 'covered_folds': 2,
 'true_positive': 15,
 'false_positive': 0,
 'false_negative': 51,
 'true_negative': 59,
 'state_precision': 1.0,
 'state_recall': 0.22727272727272727,
 'true_positive_rate': 0.22727272727272727,
 'false_positive_rate': 0.0,
 'effective_false_positive': 0,
 'effective_false_positive_rate': 0.0,
 'false_positive_lift_mean': 0.0,
 'false_positive_lift_max': 0.0,
 'target_mae': 0.14696448350712077,
 'rebound_target_mae': 0.27834182482409237,
 'non_rebound_target_mae': 0.0,
 'recipe_gate': True,
 'recipe_loss': 0.0,
 'post_event_gate': True,
 'prefer_soxx_spread': True}

In [8]:
target_semantics_gate = bool(target_lift_summary.get('training_allowed', False))
selected_gate = bool(selected and bool(selected.get('recipe_gate', False)) and bool(selected.get('post_event_gate', False)))
current_lift_rows = int(target_lift_summary.get('current_positive_rebound_lift_rows', 0))
release_lift_rows = int(target_lift_summary.get('candidate_release_positive_lift_rows', 0))

if not target_semantics_gate and selected_gate:
    decision = 'conservative_release_gate_candidate_no_training'
    next_mechanism = 'revise_target_semantics_with_sparse_soxx_spread_release_then_fold_local_bc_screen'
elif not target_semantics_gate:
    decision = 'target_lift_still_blocks_training'
    next_mechanism = 'discover_safer_release_target_before_bc_ppo'
elif selected_gate:
    decision = 'target_lift_and_release_gate_ready_for_bc_screen'
    next_mechanism = 'run_fold_local_bc_warmstart_then_walkforward_ppo'
else:
    decision = 'target_lift_ready_but_release_gate_missing'
    next_mechanism = 'screen_state_head_or_drawdown_controller'

post_event_selected = post_event_audit[
    (post_event_audit['feature_set'].astype(str) == str(selected.get('feature_set', '')))
    & (post_event_audit['threshold'].astype(float) == float(selected.get('threshold', np.nan)))
    & (post_event_audit['period'].astype(str) == 'post_event')
] if selected else pd.DataFrame()
post_event_selected_row = post_event_selected.iloc[0].to_dict() if not post_event_selected.empty else {}

selection = {
    'experiment': EXPERIMENT,
    'created_at': CREATED_AT,
    'decision': decision,
    'next_mechanism': next_mechanism,
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'fold_local_joint_reward_targets_v1',
    'prior_diagnostic': str(PRIOR_CARD.relative_to(ROOT)),
    'role': 'diagnose',
    'changed_layer': ['feature', 'model_selection', 'reward'],
    'source_targets': str(TARGETS_PATH.relative_to(ROOT)),
    'target_semantics_gate': target_semantics_gate,
    'training_allowed': False,
    'promotion_allowed': False,
    'promotion_blocker': 'No PPO training, fixed-split backtest, or walk-forward promotion review is run in this notebook.',
    'current_positive_rebound_lift_rows': current_lift_rows,
    'candidate_release_positive_lift_rows': release_lift_rows,
    'selected_feature_set': str(selected.get('feature_set', '')),
    'selected_threshold': float(selected.get('threshold', np.nan)) if selected else None,
    'selected_event_precision': float(selected.get('state_precision', np.nan)) if selected else None,
    'selected_event_recall': float(selected.get('state_recall', np.nan)) if selected else None,
    'selected_event_false_positive_rate': float(selected.get('false_positive_rate', np.nan)) if selected else None,
    'selected_event_effective_false_positive_rate': float(selected.get('effective_false_positive_rate', np.nan)) if selected else None,
    'selected_post_event_false_positive_rate': float(post_event_selected_row.get('false_positive_rate', np.nan)) if post_event_selected_row else None,
    'selected_post_event_effective_false_positive_rows': int(post_event_selected_row.get('effective_false_positive_rows', 0)) if post_event_selected_row else None,
    'selected_post_event_prediction_rate': float(post_event_selected_row.get('prediction_rate', np.nan)) if post_event_selected_row else None,
    'release_gate_candidate': selected_gate,
    'acceptance_gate_for_next_run': {
        'target_semantics_must_encode_positive_rebound_lift': True,
        'event_precision_min': 0.75,
        'event_recall_min': 0.15,
        'event_false_positive_rate_max': 0.05,
        'post_event_false_positive_rate_max': 0.05,
        'post_event_effective_false_positive_rows_max': 0,
        'fixed_split_required_after_target_revision': True,
        'walkforward_required_after_target_revision': True,
        'relative_return_min_pp': 30.0,
        'max_dd_max_pct': 15.0,
    },
    'external_sources': [
        'https://github.com/microsoft/qlib',
        'https://arxiv.org/abs/2009.11189',
        'https://github.com/AI4Finance-Foundation/FinRL',
        'https://arxiv.org/abs/2011.09607',
        'https://arxiv.org/abs/1708.02002',
        'https://arxiv.org/abs/1511.05952',
        'https://doi.org/10.21314/JCF.2016.322',
    ],
    'sidecars': [
        'target_lift_audit.csv',
        'target_lift_summary.json',
        'coverage.json',
        'event_fold_summary.csv',
        'conservative_recipe_scores.csv',
        'conservative_recipe_row_scores.csv',
        'selected_release_gate_ranking.csv',
        'post_event_false_positive_audit.csv',
        'post_event_release_row_scores.csv',
        'selection.json',
        'summary.json',
    ],
}

summary = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'conclusion': [
        f"Current target semantics still encode {current_lift_rows} positive rebound-lift rows, so direct weighted BC/PPO remains blocked.",
        f"A release-priority target has {release_lift_rows} candidate positive-lift rebound rows, so target semantics can be revised before training.",
        f"Selected release gate: {selection['selected_feature_set']} threshold {selection['selected_threshold']} with heldout precision {selection['selected_event_precision']:.3f}, recall {selection['selected_event_recall']:.3f}, and event FPR {selection['selected_event_false_positive_rate']:.3f}.",
        f"Post-event selected-gate FPR is {selection['selected_post_event_false_positive_rate']:.3f} with {selection['selected_post_event_effective_false_positive_rows']} effective false-positive release rows.",
        'Conclusion: do not train yet; next run should revise target semantics using a sparse high-precision SOXX/SOXL spread release gate, then re-run fold-local BC readiness and walk-forward validation.',
    ],
    'selection': selection,
}
write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)

for line in summary['conclusion']:
    display(Markdown(f'- {line}'))
selection


- Current target semantics still encode 0 positive rebound-lift rows, so direct weighted BC/PPO remains blocked.

- A release-priority target has 114 candidate positive-lift rebound rows, so target semantics can be revised before training.

- Selected release gate: soxx_spread_oscillation threshold 0.75 with heldout precision 1.000, recall 0.227, and event FPR 0.000.

- Post-event selected-gate FPR is 0.000 with 0 effective false-positive release rows.

- Conclusion: do not train yet; next run should revise target semantics using a sparse high-precision SOXX/SOXL spread release gate, then re-run fold-local BC readiness and walk-forward validation.

{'experiment': 'conservative_rebound_release_gate_v1',
 'created_at': '2026-05-25T17:44:55.169754+00:00',
 'decision': 'conservative_release_gate_candidate_no_training',
 'next_mechanism': 'revise_target_semantics_with_sparse_soxx_spread_release_then_fold_local_bc_screen',
 'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
 'direct_comparison_baseline': 'fold_local_joint_reward_targets_v1',
 'prior_diagnostic': 'runtime/model_selection/card/fold_local_rebound_priority_conflict_v1',
 'role': 'diagnose',
 'changed_layer': ['feature', 'model_selection', 'reward'],
 'source_targets': 'runtime/model_selection/card/fold_local_joint_reward_targets_v1/fold_local_joint_targets.csv',
 'target_semantics_gate': False,
 'training_allowed': False,
 'promotion_allowed': False,
 'promotion_blocker': 'No PPO training, fixed-split backtest, or walk-forward promotion review is run in this notebook.',
 'current_positive_rebound_lift_rows': 0,
 'candidate_release_positive_lift_rows': 114,
 'se